# Cross-source event matching

This notebook demonstrates the cross-source event pairing methodology. It is an internal working document for sharing and discussing the method: it runs every step on one real event and shows what goes in and what comes out.

**The problem.** Montandon holds records of disasters from many sources, and each source describes a disaster in its own way: USGS the magnitude, EM-DAT the losses, IDMC the displacement, CEMS the mapping. No shared identifier reliably says that two records are about the same disaster, so the records cannot simply be looked up and joined.

**The method, in five steps.** Each step is a section below.

1. **Start from one event.** The event selected when the run was started.
2. **Decide where and when to look.** A time window and a radius around the event. Both depend on the type of hazard.
3. **Collect candidates.** Every record from the other sources that falls inside that window.
4. **Ask Jev.** Jev is a decision model. For each candidate it gives the probability that the candidate describes the same disaster as the selected event.
5. **Compare.** What the matched records report goes into a fact sheet, a chart and a map.

**Terms used below.**

- **Record**: one entry in one source, for example one USGS earthquake or one EM-DAT disaster.
- **Selected event**: the record the matching starts from.
- **Candidate**: a record from another source that is close enough in time and place to be worth checking.
- **P(same)**: Jev's probability that a candidate describes the same disaster as the selected event.
- **Joined**: a candidate whose P(same) is at or above the confidence threshold (default 0.6).
- **Possible**: a candidate with P(same) between 0.3 and the threshold.
- **Exact link**: a link that Montandon itself holds between two records. These join without asking Jev.

The scores are model estimates. See *Limitations* at the end.

In [ ]:
# --- Reserved parameters (injected by the Notebook Factory; defaults keep the notebook runnable standalone) ---
run_id = ""
inputs_dir = ""
output_dir = "outputs"
area = {}                      # {id, code, name, level, country, country_name, bbox}
area_geojson = ""              # FeatureCollection with the target area
subdivisions_geojson = ""      # FeatureCollection with the aggregation subdivisions
stac_item = {}                 # the selected Montandon event (dict)
stac_item_json = ""
stac_collection = ""
stac_api_url = ""              # root of the item's catalog (Montandon)
stac_catalog_profile = ""      # "generic" | "montandon"
filters = {}                   # the run's filter tags
hazard = ""

# --- User parameters ---
confidence_threshold = 0.6     # a candidate joins the event at this probability or more (possible: 0.3 up to it)
max_candidates = 60            # most candidates to judge, the most plausible by time and distance
hazard_windows = """EQ 7 300
TS 7 1500
VO 30 200
LS 7 100
AV 7 100
FL 30 500
FF 14 300
TC 21 2500
ST 14 800
SS 14 800
DR 180 2000
WF 30 300
HT 30 1500
CW 30 1500
EP 90 3000"""                  # one hazard per line: GLIDE code, days either side of the event, radius in km
default_window_days = 14       # window for a hazard that is not in the table
default_radius_km = 500        # radius for a hazard that is not in the table
coarse_radius_km = 1500        # at least this radius when a record is only located at country level
min_cross_hazard_magnitude = 5.0   # a smaller earthquake (no tsunami flag) is only compared with earthquakes


## 1. The selected event

This is the record the matching starts from. Every other record is compared with it, so its source, hazard type, date and countries decide where and when the search looks.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from manywidgets import Chart, Column, Legend, RangeSlider, Row, Stat

sys.path.insert(0, str(Path.cwd()))  # the helper module ships next to the notebook
import matching as M

api_url = stac_api_url or os.environ.get("STAC_API_URL", "")
token = os.environ.get("STAC_API_KEY", "")
if not stac_item or not api_url:
    raise RuntimeError("Select a Montandon event when starting the run: it is the event to match.")

seed = M.record_from_item(stac_item)
rules = M.Rules.from_params(
    hazard_windows, default_window_days, default_radius_km, coarse_radius_km, min_cross_hazard_magnitude
)

Row(
    Stat(label="Source", value=M.SOURCE_LABELS.get(seed.source, seed.source)),
    Stat(label="Hazard", value=M.HAZARD_LABELS.get(seed.glide, "unknown").capitalize()),
    Stat(label="Event", value=seed.title),
    Stat(label="Date", value=seed.start.date().isoformat() if seed.start else "unknown"),
    Stat(label="Countries", value=", ".join(seed.countries) or "unknown"),
    gap="16px",
)


## 2. Where and when to look

Records of the same disaster rarely carry the same date or place. A register such as EM-DAT dates a disaster to a day and places it in a country. A satellite activation can start a day or more later. A seismic catalogue gives an exact point. To allow for this the search uses a **window** around the selected event: a number of days before and after its dates, and a radius in kilometres around its location.

The window depends on the hazard. An earthquake is over within days, a flood or a cyclone unfolds over weeks, and a drought over months. The table below is the rule set used. It is a run-form parameter, and the row for this event's hazard is highlighted. Records that a source only places at country level are kept when they share a country with the selected event, whatever their distance.

In [ ]:
window = M.search_window(seed, rules)
display(Row(
    Stat(label="Search from", value=f"{window.first:%d %b %Y}"),
    Stat(label="Search to", value=f"{window.last:%d %b %Y}"),
    Stat(label="Window", value=f"\u00b1 {window.days} days"),
    Stat(label="Radius", value=f"{window.radius_km} km"),
    gap="16px",
))
print(
    f"The selected event started on {seed.start:%d %b %Y}, so the search accepts records that start between "
    f"{window.first:%d %b %Y} and {window.last:%d %b %Y}."
)
if window.countries:
    print("Records placed in " + ", ".join(window.countries) + " are also asked for, wherever in the country they are.")

rule_rows = pd.DataFrame(
    [{"Hazard": M.HAZARD_LABELS.get(code_, code_).capitalize(), "Code": code_, "Days either side": d, "Radius (km)": km}
     for code_, (d, km) in rules.windows.items()]
)
(
    rule_rows.style.apply(lambda row: ["background-color: #fbd9b8" if row["Code"] == seed.glide else "" for _ in row], axis=1)
    .set_properties(**{"text-align": "left"})
    .set_table_styles([{"selector": "th", "props": "text-align: left;"}])
    .hide(axis="index")
)

## 3. Candidates

The catalogue is asked for every record, from every source, that starts inside the window and is either in the same country or inside the box around the selected event. A record carrying the same Montandon correlation id as the selected event is always added, even outside the window.

Records from the selected event's own source are left out unless Montandon links them to it. When a source files several episodes of one event, only the one closest in time and distance is kept. The rest are ranked by how close they are in time and distance and cut to the number set in the run form (`max_candidates`), so a busy window can hide a record.

**Nothing is judged here.** A candidate is only a record worth asking about. The table shows how far in days and kilometres each one is from the selected event, and why it was kept.

In [ ]:
candidates, searched = M.find_candidates(api_url, token, seed, max_candidates, rules)
exact = M.exact_links(seed, candidates)  # records Montandon links to the event: they skip Jev
by_id = {c.id: c for c in candidates}

display(Row(
    Stat(label="Records in the window", value=searched),
    Stat(label="Candidates kept", value=len(candidates)),
    Stat(label="Sources", value=len({c.source for c in candidates})),
    Stat(label="Exact links", value=len(exact)),
    gap="16px",
))
if searched > len(candidates):
    print(f"{searched - len(candidates)} further records were in the window; only the {len(candidates)} closest are kept.")
(
    M.candidate_table(seed, candidates, exact, rules)
    .style.set_properties(**{"text-align": "left"})
    .set_properties(subset=["Date"], **{"white-space": "nowrap"})
    .set_table_styles([{"selector": "th", "props": "text-align: left;"}])
    .hide(axis="index")
    if candidates else print("No other source has a record in this window.")
)

### The candidates on a map

The grey box is the area asked of the catalogue. The orange circle is the hazard's radius, applied to records with a precise location. Each dot is a candidate, coloured by its source, and the selected event has a larger dot with a black outline. Records that a source places only at country level are drawn at the single point the source gives.

In [ ]:
import geopandas as gpd
import numpy as np
from lonboard import Map, PolygonLayer, ScatterplotLayer
from lonboard.layer_extension import DataFilterExtension
from manywidgets.lonboard import FilterBinder, LayerToggle
from shapely.geometry import box, shape


def rgb(hex_color, alpha=230):
    h = hex_color.lstrip("#")
    return [int(h[i : i + 2], 16) for i in (0, 2, 4)] + [alpha]


if seed.point is None or window.bbox is None:
    print("The selected event has no location, so there is nothing to draw.")
else:
    placed = [(seed, "selected event")] + [(c, "candidate") for c in candidates if c.point]
    labels = [M.SOURCE_LABELS.get(rec.source, rec.source) for rec, _ in placed]
    gdf = gpd.GeoDataFrame(
        {"title": [rec.title for rec, _ in placed], "source": labels,
         "date": [rec.start.date().isoformat() if rec.start else "" for rec, _ in placed],
         "role": [role for _, role in placed]},
        geometry=gpd.points_from_xy([rec.point[0] for rec, _ in placed], [rec.point[1] for rec, _ in placed]),
        crs="EPSG:4326",
    )
    seed_flags = np.arange(len(placed)) == 0
    area_layers = [
        PolygonLayer.from_geopandas(
            gpd.GeoDataFrame({"name": ["Area asked of the catalogue"]}, geometry=[box(*window.bbox)], crs="EPSG:4326"),
            get_fill_color=[120, 120, 120, 18], get_line_color=[90, 90, 90, 220], line_width_min_pixels=2, stroked=True,
        ),
        PolygonLayer.from_geopandas(
            gpd.GeoDataFrame({"name": [f"{window.radius_km} km radius"]},
                             geometry=[shape(M.circle_polygon(seed.point, window.radius_km))], crs="EPSG:4326"),
            get_fill_color=[213, 94, 0, 25], get_line_color=[213, 94, 0, 230], line_width_min_pixels=2, stroked=True,
        ),
    ]
    dots = ScatterplotLayer.from_geopandas(
        gdf,
        get_fill_color=np.array([rgb(M.COLOR_BY_LABEL.get(label, "#777777")) for label in labels], dtype=np.uint8),
        get_line_color=np.where(seed_flags[:, None], [0, 0, 0, 255], [255, 255, 255, 220]).astype(np.uint8),
        get_radius=np.where(seed_flags, 12, 7), radius_units="pixels", stroked=True, line_width_min_pixels=2,
        pickable=True, auto_highlight=True,
    )
    west, south, east, north = window.bbox
    view = M.map_view([(west, south), (east, north)])
    legend = Legend(entries=[[M.COLOR_BY_LABEL.get(s, "#777777"), s] for s in dict.fromkeys(labels)], title="Source")
    toggles = [LayerToggle(layer=area_layers[0], label="Search box"), LayerToggle(layer=area_layers[1], label="Radius")]
    display(Column(Row(legend, *toggles, gap="24px"),
                   Map([*area_layers, dots], show_tooltip=True, height=520, view_state=view), gap="12px"))
    if len(placed) - 1 < len(candidates):
        print(f"{len(candidates) - len(placed) + 1} candidates have no location and are not drawn.")

## 4. Asking Jev

Jev is a decision model: it answers a question about some data with a probability instead of a plain yes or no. Here the question is *"Does this candidate describe the same disaster as the selected event?"*, asked once for every candidate.

**What is sent.**

- Candidates are sent one source at a time, at most 25 per request, so Jev compares a candidate with the other records of its own source. That helps when a source holds several records near the selected event.
- Each request holds the **selected event** (called the *anchor*) and the **candidates**, each described in words and facts: the kind of record, title, description, hazard type, start and end date, magnitude when the record has one, and the countries.
- The days and kilometres between the candidate and the selected event are measured here and passed along. Jev takes them as given.
- Jev does **not** see record ids, codes, impact figures or links.
- Alongside the yes/no question for each candidate there are two more: which candidate is the best match, and whether any candidate matches at all.

**What comes back.** One probability per candidate, called P(same). Exact links are not sent: they join at 100% because Montandon itself links them.

The next cell shows what a request contains. The one after sends them.

In [ ]:
import json

to_ask = [c for c in candidates if c.id not in exact]
groups = M.jev_groups(to_ask)
display(Row(
    Stat(label="Candidates sent to Jev", value=len(to_ask)),
    Stat(label="Requests", value=len(groups)),
    Stat(label="Skipped (exact link)", value=len(exact)),
    gap="16px",
))
if groups:
    print("Requests, one source at a time:")
    for group in groups:
        print(f"  {M.SOURCE_LABELS.get(group[0].source, group[0].source):12s} {len(group)} candidates")
    closest = to_ask[0]  # the candidates are ranked by closeness, so this is the closest one
    group = next(g for g in groups if closest in g)
    key = f"candidate_{group.index(closest) + 1}"  # its name inside the request
    request = M.listwise_request(M.DEFAULT_MODEL, seed, group, rules)
    example = request["questions"][f"same_{key}"]
    print("\nThe selected event, as Jev reads it:")
    print(json.dumps(request["state"]["anchor"], indent=2, ensure_ascii=False))
    print("\nThe closest candidate, as Jev reads it (the last two lines are measured here, not read from the record):")
    print(json.dumps(request["state"]["candidates"][key], indent=2, ensure_ascii=False))
    print("\nThe question asked about that candidate:")
    print(example["instructions"])
    print("  true :", example["criteria"]["true"])
    print("  false:", example["criteria"]["false"])
else:
    print("There is nothing to send to Jev: every candidate is an exact link, or there are no candidates.")

### Jev's answers

The requests are sent now. The result is one row per candidate with its P(same), sorted from most to least likely, in the table below. A candidate at or above the confidence threshold is **joined**. One between 0.3 and the threshold is **possible**. The threshold is a run-form parameter, and the map further down has a slider to change it afterwards.

The cost of the calls is printed under the counts. Without an `OPENROUTER_API_KEY` secret nothing is sent: the candidates are listed without a score, and only exact links join.

In [ ]:
api_key = os.environ.get("OPENROUTER_API_KEY", "")
jev = M.ask_jev(seed, to_ask, api_key, rules=rules)
matches = M.match_table(candidates, exact, jev.scores)

if not api_key:
    print("No OPENROUTER_API_KEY: the candidates are not scored, and only exact links join the event.")
for error in jev.errors:
    print("A Jev call failed, its candidates have no score:", error)

joined = matches[matches.p_same >= confidence_threshold]
possible = matches[(matches.p_same >= 0.3) & (matches.p_same < confidence_threshold)]
display(Row(
    Stat(label="Candidates", value=len(matches)),
    Stat(label=f"Joined (P at least {confidence_threshold:g})", value=len(joined)),
    Stat(label=f"Possible (0.3 to {confidence_threshold:g})", value=len(possible)),
    gap="16px",
))
if api_key:
    print(f"Jev scored {len(jev.scores)} of {len(to_ask)} candidates in {len(groups)} requests, cost ${jev.cost:.4f}.")
(
    matches.drop(columns="id")
    .rename(columns={"source": "Source", "title": "Title", "date": "Date", "p_same": "P(same)", "why": "Decided by"})
    .style.format({"P(same)": "{:.0%}"})
    .background_gradient(subset=["P(same)"], cmap="Greens", vmin=0, vmax=1)
    .set_properties(**{"text-align": "left"})
    .set_properties(subset=["Date"], **{"white-space": "nowrap"})
    .set_table_styles([{"selector": "th", "props": "text-align: left;"}])
    .hide(axis="index")
    if len(matches) else print("No other source has a record in this window.")
)

## 5. Fact sheet

The joined records are now treated as one disaster. The fact sheet puts what each source reports side by side: one column per source, one row per measure.

- **Orange cells** mark rows where the sources disagree: the largest figure is at least 1.5 times the smallest.
- An **asterisk** (\*) marks a modelled figure: an estimate by the source (USGS PAGER), not a report.
- A source can hold several records of the disaster. Copernicus figures are summed over its areas of interest, a register's records for different countries (EM-DAT, GLIDE) are added up, and otherwise the largest figure is shown.
- A disagreement is a result, not an error. Sources measure different things, so the sheet shows the differences and does not choose between them.

In [ ]:
events = [seed] + [by_id[i] for i in joined.id]
details = M.fetch_details(api_url, token, events)
facts = M.fact_sheet(events, details.rows)
M.display_table(facts)


## 6. Figures by source

The same numbers as bars, one bar per source, for deaths, people affected and people displaced. Bars that differ a lot show the sources disagreeing. A source with no figure for a measure has no bar. Modelled figures are labelled.

In [ ]:
def metric_chart(label, title):
    columns = [c for c in facts.columns[:-2] if label in facts.index and pd.notna(facts.loc[label, c])]
    if not columns:
        return None
    chart = Chart(chart_type="bar", title=title, width=700, height=300)
    chart.set_options(scales={"x": {"type": "category"}, "y": {"beginAtZero": True}})
    for col in columns:
        name = col + (" (modelled)" if (label, col) in facts.attrs["modelled"] else "")
        chart.add_series(x=[label], y=[float(facts.loc[label, col])], name=name, series_type="bar",
                         color=M.COLOR_BY_LABEL.get(col))
    return chart


charts = [c for c in (metric_chart("Deaths", "Deaths"), metric_chart("People affected", "People affected"),
                      metric_chart("People displaced", "People displaced")) if c]
chart_view = Column(*charts, gap="12px") if charts else None
if chart_view is None:
    print("No joined source reported deaths, people affected or people displaced.")
chart_view


## 7. Map of the matched records

Every candidate, coloured by source. The slider sets the minimum confidence: records appear and disappear as it moves, including in the published page, so the effect of the threshold can be seen directly. The selected event has a larger marker. Footprints are precise areas from the sources (Copernicus EMS areas, shake areas), never country outlines, and can be switched on and off.

In [ ]:
located = [(seed, 1.0, "selected event")] + [
    (by_id[r.id], r.p_same, r.why) for r in matches.itertuples() if by_id[r.id].point
]
labels = [M.SOURCE_LABELS.get(rec.source, rec.source) for rec, _, _ in located]
points_gdf = gpd.GeoDataFrame(
    {"title": [rec.title for rec, _, _ in located], "source": labels,
     "P(same)": [round(p, 2) for _, p, _ in located], "why": [why for _, _, why in located]},
    geometry=gpd.points_from_xy([rec.point[0] for rec, _, _ in located], [rec.point[1] for rec, _, _ in located]),
    crs="EPSG:4326",
)
is_seed = np.arange(len(located)) == 0
points = ScatterplotLayer.from_geopandas(
    points_gdf,
    get_fill_color=np.array([rgb(M.COLOR_BY_LABEL.get(label, "#777777")) for label in labels], dtype=np.uint8),
    get_line_color=np.where(is_seed[:, None], [0, 0, 0, 255], [255, 255, 255, 220]).astype(np.uint8),
    get_radius=np.where(is_seed, 12, 7),
    radius_units="pixels", stroked=True, line_width_min_pixels=2,
    pickable=True, auto_highlight=True,
    extensions=[DataFilterExtension(filter_size=1)],
    get_filter_value=points_gdf["P(same)"].to_numpy(dtype="float32"),
    filter_range=(confidence_threshold, 1.0),
)

shapes = M.footprints(events, details)
layers, controls = [], []
if shapes:
    shape_gdf = gpd.GeoDataFrame.from_features(shapes, crs="EPSG:4326")[["source", "title", "geometry"]]
    footprint_layer = PolygonLayer.from_geopandas(
        shape_gdf, get_fill_color=[213, 94, 0, 45], get_line_color=[213, 94, 0, 230],
        line_width_min_pixels=1, stroked=True, pickable=True,
    )
    layers.append(footprint_layer)
    controls.append(LayerToggle(layer=footprint_layer, label="Precise footprints"))
layers.append(points)

# open on the records that joined, not on every candidate (the filter hides the rest)
view = M.map_view([(rec.point[0], rec.point[1]) for rec, p, _ in located if p >= confidence_threshold])

slider = RangeSlider(min=0, max=1, low=confidence_threshold, high=1.0, step=0.05,
                     label="Minimum confidence: probability that the record is the same disaster")
binder = FilterBinder(slider, points)
legend = Legend(entries=[[M.COLOR_BY_LABEL.get(s, "#777777"), s] for s in dict.fromkeys(labels)], title="Source")
Column(Row(legend, *controls, gap="24px"), slider, binder, Map(layers, show_tooltip=True, height=520, view_state=view), gap="12px")


## Limitations

- **Scores are model estimates.** Jev reads titles, dates, places and magnitudes and returns a probability. It can be wrong in both directions. The default confidence threshold is 0.6.
- **Aftershocks.** Earthquakes close to the selected event in time and place, such as aftershocks, are not handled separately. Jev can score a small aftershock as highly as the main shock, and they appear among the candidates. How to treat them is an open question for subject matter experts.
- **Search scope.** The search starts from the selected record. Records outside the time window and radius are not found; both are run-form parameters. Other records of the selected event's own source are not candidates unless Montandon links them. Candidates are capped at the most plausible by time and distance, so a very busy window can hide a record.
- **Exact links.** Only Montandon's `related` event links join without Jev, and few exist so far (CEMS to GDACS and Charter). A record with the selected event's `monty:corr_id` is always a candidate, including outside the time window and radius and first when the list is capped, but it is scored like any other. A correlation id stands for one date, one country and one hazard type: it can cover separate events of the same day, and it does not cover a disaster's records in other countries or on other dates. No link is inferred from an id or a register number.
- **Disagreement.** Differing figures are not treated as errors. Sources measure different things: reported against modelled, deaths against killed or missing, a country against an area of interest. Modelled figures carry an asterisk. EM-DAT publishes damage in thousands of US dollars, but its catalogue unit says only "count", so amounts in different units are never compared.